In [ ]:
import sys
sys.path.append("/app")

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from utils import create_spark_session, load_config

spark = create_spark_session("silver_ordem_compras")
config = load_config()

bronze_path = config["storage"]["bronze"]["path"]
silver_path = config["storage"]["silver"]["path"]

df = spark.read.format("delta").load(f"{bronze_path}/ordem_compras")

df_silver = df \
    .withColumnRenamed("NOME", "nome_produto") \
    .withColumnRenamed("FORNECEDOR_ID", "fornecedor_id") \
    .withColumn("prazo_entrega", F.to_date("prazo_entrega")) \
    .withColumn("valor_unitario", F.col("valor_unitario").cast(DecimalType(15, 2))) \
    .withColumn("valor_total", F.col("valor_total").cast(DecimalType(15, 2))) \
    .withColumn("quantidade", F.col("quantidade").cast("integer")) \
    .withColumn("saldo", F.col("saldo").cast("integer")) \
    .withColumn("status", F.trim(F.lower(F.col("status")))) \
    .withColumn("razao_social", F.trim(F.col("razao_social"))) \
    .withColumn("nome_produto", F.trim(F.col("nome_produto")))

df_silver.write.format("delta").mode("overwrite").save(f"{silver_path}/ordem_compras")

print(f"Silver salvo em {silver_path}/ordem_compras")
spark.stop()